# 02 · Tamanho do município e a queda do PT (hipótese A1)

**Teste confirmatório.** A especificação foi registrada em `docs/hipoteses.md` (seção A1)
antes de qualquer regressão, e é seguida aqui sem mudanças.

**A pergunta:** o PT perdeu mais espaço entre 2022 e 2026 nos municípios menores porque
eles são pequenos, ou porque municípios pequenos têm outra composição (região, renda,
religião, agro, idade…)?

**O teste:** regredir a variação do PT em log(eleitorado de 2022), acrescentando em
etapas efeitos fixos geográficos e controles de composição. Se o coeficiente do tamanho
continuar positivo (maior → perde menos) com tudo controlado, a hipótese se sustenta.

| | |
|---|---|
| Unidade | AMC (5.569), 1º turno |
| Desfecho | Δ logit(`pt_2p`) 2022 → 2026; em pp como robustez |
| Regressor | log(eleitores aptos em 2022) |
| Principal (M3) | MQO, efeitos fixos de região imediata, erros agrupados por região imediata, 9 controles |
| Critério | A1 se sustenta se o coeficiente em M3 for positivo com IC 95% acima de zero |

In [ ]:
import warnings

import altair as alt
import numpy as np
import polars as pl
import pyfixest as pf

from kingsrace import paths
from kingsrace.features.base import base_variacao
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import SERIES, TINTA

alt.renderers.enable("jupyter", offline=True)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*singleton fixed effect")  # explicado abaixo

P = paths.PROCESSED
base = base_variacao(
    pl.read_parquet(P / "painel_amc.parquet"),
    pl.read_parquet(P / "covariaveis_amc.parquet"),
    pl.read_parquet(P / "municipios.parquet"),
    de=(2022, 1),
    para=(2026, 1),
).with_columns(
    log_renda=pl.col("renda_pc_mediana").log(),
    log_pop=pl.col("populacao").log(),
    nordeste=(pl.col("regiao") == "NE").cast(pl.Int8),
)

## Amostra e variáveis

Os controles registrados. Todos os modelos usam **a mesma amostra** (AMCs com todas as
variáveis de M3 preenchidas): assim, a mudança do coeficiente de M0 para M3 reflete só os
controles, não uma troca de amostra.

In [ ]:
CONTROLES = [
    "pt_2p_logit_de",  # nível do PT em 2022: teto e regressão à média
    "pct_urbana",
    "idade_mediana",
    "pct_evangelica",
    "log_renda",  # log da renda domiciliar per capita mediana
    "taxa_alfabetizacao",
    "pct_preta_parda",
    "pct_va_agro_2021",
    "pct_renda_outras_fontes",  # aposentadorias, programas sociais etc.
]
amostra = base.drop_nulls(["delta_logit", "log_aptos_de", *CONTROLES]).to_pandas()
print(f"AMCs: {base.height} | amostra completa: {len(amostra)}")

## Modelos

A sequência registrada: M0 bruto → M1 + efeitos fixos de UF → M2 + efeitos fixos de região
imediata (510 regiões) → M3 + controles. Depois, as checagens de robustez.

O `pyfixest` descarta grupos de efeito fixo com um único município (por exemplo, o DF com
efeitos fixos de UF), então o *n* varia em 1 ou 2.

In [ ]:
cluster = {"CRV1": "cd_regiao_imediata"}
ctrl = " + ".join(CONTROLES)


def ajusta(formula, dados=amostra, **kw):
    return pf.feols(formula, dados, vcov=cluster, **kw)


modelos = {
    "M0 bruto": ajusta("delta_logit ~ log_aptos_de"),
    "M1 + EF de UF": ajusta("delta_logit ~ log_aptos_de | uf"),
    "M2 + EF de região imediata": ajusta("delta_logit ~ log_aptos_de | cd_regiao_imediata"),
    "M3 + controles": ajusta(f"delta_logit ~ log_aptos_de + {ctrl} | cd_regiao_imediata"),
}
robustez = {
    "R2 pesos pelo eleitorado": ajusta(
        f"delta_logit ~ log_aptos_de + {ctrl} | cd_regiao_imediata", weights="aptos_de"
    ),
    "R3 EF de UF": ajusta(f"delta_logit ~ log_aptos_de + {ctrl} | uf"),
    "R4 tamanho pela população": ajusta(f"delta_logit ~ log_pop + {ctrl} | cd_regiao_imediata"),
    "R5 sem AMCs < 2 mil eleitores": ajusta(
        f"delta_logit ~ log_aptos_de + {ctrl} | cd_regiao_imediata",
        dados=amostra[amostra.aptos_de >= 2000],
    ),
}
em_pp = ajusta(f"delta_pp ~ log_aptos_de + {ctrl} | cd_regiao_imediata")

pf.etable(list(modelos.values()), keep=["log_aptos_de"], type="df")

### Coeficiente do tamanho em cada modelo

**Como ler:** cada ponto é o coeficiente de log(eleitorado) num modelo; a barra é o IC de
95%. Acima de zero = municípios maiores perderam menos. Os quatro primeiros são a sequência
confirmatória; os demais, checagens de robustez (só qualificam o veredito).

In [ ]:
def coeficiente(nome, fit, grupo):
    var = "log_pop" if "log_pop" in fit.tidy().index else "log_aptos_de"
    t = fit.tidy().loc[var]
    return {
        "modelo": nome,
        "grupo": grupo,
        "beta": t["Estimate"],
        "ic_inf": t["2.5%"],
        "ic_sup": t["97.5%"],
        "n": fit._N,
    }


coefs = pl.DataFrame(
    [coeficiente(n, f, "Sequência registrada") for n, f in modelos.items()]
    + [coeficiente(n, f, "Robustez") for n, f in robustez.items()]
)
ordem = coefs["modelo"].to_list()
cores_grupo = alt.Scale(domain=["Sequência registrada", "Robustez"], range=SERIES[:2])
base_coef = alt.Chart(coefs).encode(
    y=alt.Y("modelo:N", sort=ordem, title=None),
    color=alt.Color("grupo:N", scale=cores_grupo, title=None),
    tooltip=[
        alt.Tooltip("modelo:N", title="Modelo"),
        alt.Tooltip("beta:Q", title="Coeficiente", format="+.4f"),
        alt.Tooltip("ic_inf:Q", title="IC 95% inf.", format="+.4f"),
        alt.Tooltip("ic_sup:Q", title="IC 95% sup.", format="+.4f"),
        alt.Tooltip("n:Q", title="AMCs"),
    ],
)
barras = base_coef.mark_rule(strokeWidth=2).encode(
    x=alt.X("ic_inf:Q", title="Coeficiente de log(eleitorado) sobre Δ logit do PT"),
    x2="ic_sup:Q",
)
pontos = base_coef.mark_circle(size=80, opacity=1).encode(x="beta:Q")
zero = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(x=alt.datum(0))
(zero + barras + pontos).properties(
    width=520, height=260, title="O efeito do tamanho cai pela metade com controles, mas não some"
)

### Antes e depois de controlar

**Como ler:** cada ponto é a média de 1/25 das AMCs, agrupadas pelo tamanho. Em azul, a
relação bruta (M0); em laranja, a relação **depois de descontar** região imediata e
controles (regressão parcial de M3: tamanho e queda são substituídos pelo que sobra
deles depois dos controles). As duas estão centradas em zero; a inclinação de cada linha é
o coeficiente do modelo correspondente.

In [ ]:
ajuste_parcial = pf.feols(
    f"delta_logit + log_aptos_de ~ {ctrl} | cd_regiao_imediata", amostra, vcov=cluster
)
res_y, res_x = (m.resid() for m in ajuste_parcial.all_fitted_models.values())
bruto_x = amostra.log_aptos_de - amostra.log_aptos_de.mean()
bruto_y = amostra.delta_logit - amostra.delta_logit.mean()


def binscatter(x, y, rotulo, n_bins=25):
    df = pl.DataFrame({"x": np.asarray(x), "y": np.asarray(y)})
    return (
        df.with_columns(bin=pl.col("x").rank("ordinal").mul(n_bins).floordiv(df.height + 1))
        .group_by("bin")
        .agg(pl.col("x").mean(), pl.col("y").mean(), pl.len().alias("amcs"))
        .with_columns(relacao=pl.lit(rotulo))
    )


bins = pl.concat(
    [binscatter(bruto_x, bruto_y, "Bruta (M0)"), binscatter(res_x, res_y, "Condicional (M3)")]
)
retas = pl.DataFrame(
    {
        "relacao": ["Bruta (M0)"] * 2 + ["Condicional (M3)"] * 2,
        "x": [-3.0, 3.0, -3.0, 3.0],
    }
).with_columns(
    y=pl.col("x")
    * pl.when(pl.col("relacao") == "Bruta (M0)")
    .then(modelos["M0 bruto"].coef()["log_aptos_de"])
    .otherwise(modelos["M3 + controles"].coef()["log_aptos_de"])
)
cores_rel = alt.Scale(domain=["Bruta (M0)", "Condicional (M3)"], range=SERIES[:2])
eixo_x = alt.X(
    "x:Q",
    title="log(eleitorado), relativo ao esperado (0 = típico; +1 ≈ 2,7× maior)",
    scale=alt.Scale(domain=[-3, 3]),
)
eixo_y = alt.Y("y:Q", title="Δ logit do PT, relativo ao esperado")
pontos_bins = (
    alt.Chart(bins)
    .mark_circle(size=70, opacity=0.9)
    .encode(
        x=eixo_x,
        y=eixo_y,
        color=alt.Color("relacao:N", scale=cores_rel, title=None),
        tooltip=[
            alt.Tooltip("relacao:N", title="Relação"),
            alt.Tooltip("x:Q", title="Tamanho (log, relativo)", format="+.2f"),
            alt.Tooltip("y:Q", title="Δ logit (relativo)", format="+.3f"),
            alt.Tooltip("amcs:Q", title="AMCs no grupo"),
        ],
    )
)
linhas = (
    alt.Chart(retas)
    .mark_line()
    .encode(x=eixo_x, y=eixo_y, color=alt.Color("relacao:N", scale=cores_rel))
)
(pontos_bins + linhas).properties(
    width=560, height=320, title="Tamanho × queda do PT: relação bruta e descontada a composição"
)

## Em pontos percentuais

Para traduzir: o mesmo modelo M3 com o desfecho em pp (checagem R1). Comparando duas AMCs
**da mesma região imediata e com a mesma composição**, uma com 3 mil e outra com 300 mil
eleitores (100× maior, Δlog = 4,6):

In [ ]:
beta_pp = em_pp.coef()["log_aptos_de"]
ic_pp = em_pp.tidy().loc["log_aptos_de", ["2.5%", "97.5%"]]
dlog = np.log(100)
beta_pp_bruto = ajusta("delta_pp ~ log_aptos_de").coef()["log_aptos_de"]
ic_txt = f"IC 95% {ic_pp.iloc[0]:+.3f} a {ic_pp.iloc[1]:+.3f}"
print(f"Coeficiente M3 em pp: {beta_pp:+.3f} pp por unidade de log ({ic_txt})")
print(f"3 mil vs 300 mil eleitores, M3 (com controles): {beta_pp * dlog:.1f} pp a mais de queda")
print(
    f"3 mil vs 300 mil eleitores, bruta:              {beta_pp_bruto * dlog:.1f} pp a mais de queda"
)

## A1b — o Nordeste é diferente? (secundária, sugerida pela exploração)

No `01_panorama` o gradiente parecia mais fraco no Nordeste. Teste: interação
log(eleitorado) × Nordeste em M3. **Não confirmatório**, porque a ideia veio de olhar os
dados.

In [ ]:
a1b = ajusta(f"delta_logit ~ log_aptos_de + log_aptos_de:nordeste + {ctrl} | cd_regiao_imediata")
a1b.tidy().loc[["log_aptos_de", "log_aptos_de:nordeste"], ["Estimate", "2.5%", "97.5%", "Pr(>|t|)"]]

A interação é praticamente zero: **descontada a composição, o gradiente do Nordeste é o
mesmo do resto do país.** O "Nordeste plano" do panorama era efeito de composição.

## O que mais pesa na composição (exploratório)

Os coeficientes dos controles em M3, **padronizados** (efeito de 1 desvio-padrão de cada
variável sobre Δ logit), para comparar magnitudes. **Isto não é confirmatório:** os
controles estão no modelo para isolar o tamanho, e cada um deles merece sua própria
hipótese registrada (A2 agro, A3 religião, A4 transferências).

**Como ler:** à direita de zero, a variável está associada a uma queda **menor** do PT;
à esquerda, a uma queda **maior**.

In [ ]:
m3 = modelos["M3 + controles"]
dp = amostra[["log_aptos_de", *CONTROLES]].std()
nomes = {
    "log_aptos_de": "Tamanho: log(eleitorado)",
    "pt_2p_logit_de": "PT em 2022 (logit)",
    "pct_urbana": "% urbana",
    "idade_mediana": "Idade mediana",
    "pct_evangelica": "% evangélica",
    "log_renda": "Renda per capita (log)",
    "taxa_alfabetizacao": "Alfabetização",
    "pct_preta_parda": "% preta ou parda",
    "pct_va_agro_2021": "% do VA no agro",
    "pct_renda_outras_fontes": "% renda de outras fontes",
}
padron = pl.DataFrame(
    [
        {
            "variavel": nomes[v],
            "efeito": m3.tidy().loc[v, "Estimate"] * dp[v],
            "ic_inf": m3.tidy().loc[v, "2.5%"] * dp[v],
            "ic_sup": m3.tidy().loc[v, "97.5%"] * dp[v],
            "papel": "Testado (A1)" if v == "log_aptos_de" else "Controle (exploratório)",
        }
        for v in nomes
    ]
).sort("efeito")
base_p = alt.Chart(padron).encode(
    y=alt.Y("variavel:N", sort=padron["variavel"].to_list(), title=None),
    tooltip=[
        alt.Tooltip("variavel:N", title="Variável"),
        alt.Tooltip("efeito:Q", title="Efeito de 1 dp (Δ logit)", format="+.4f"),
        alt.Tooltip("ic_inf:Q", title="IC 95% inf.", format="+.4f"),
        alt.Tooltip("ic_sup:Q", title="IC 95% sup.", format="+.4f"),
    ],
)
cores_papel = alt.Scale(
    domain=["Testado (A1)", "Controle (exploratório)"], range=[SERIES[1], SERIES[0]]
)
cor_papel = alt.Color("papel:N", scale=cores_papel, title=None)
(
    zero
    + base_p.mark_rule(strokeWidth=2).encode(
        x=alt.X("ic_inf:Q", title="Efeito de 1 desvio-padrão sobre Δ logit do PT"),
        x2="ic_sup:Q",
        color=cor_papel,
    )
    + base_p.mark_circle(size=80, opacity=1).encode(x="efeito:Q", color=cor_papel)
).properties(width=520, height=300, title="Controles de M3, padronizados (exploratório)")

## Veredito

- **A1 se sustenta.** Em M3 o coeficiente do tamanho é positivo, com IC 95% bem acima de
  zero, e as quatro checagens de robustez concordam.
- **Metade do gradiente bruto é composição** (o coeficiente cai de ~0,049 em M0 para ~0,024
  em M3); a outra metade sobrevive a região imediata, renda, religião, idade, agro,
  escolaridade, cor ou raça e transferências.
- **Ordem de grandeza:** entre duas AMCs comparáveis da mesma região, uma com 3 mil e outra
  com 300 mil eleitores, a menor perdeu cerca de 2,5 pp a mais.
- **A1b não se confirma:** o Nordeste tem o mesmo gradiente quando a composição é levada em
  conta.

**O que o tamanho ainda representa?** O que sobra é "ser pequeno" depois de descontar o
que medimos, e pode ser algo que não medimos: redes locais, ambiente de informação,
dependência de emprego público, presença de lideranças locais (A5) etc. Isso não é causal:
é uma associação que sobrevive aos controles disponíveis.

**Exploratório, para registrar como hipóteses próprias:** com o resto fixo, o PT caiu
**mais** onde era mais forte em 2022 (teto/regressão à média) e onde a renda e o peso do
agro são maiores; caiu **menos** onde há mais renda de transferências e aposentadorias e,
curiosamente, onde há mais evangélicos (condicional ao voto de 2022, que já capturava o
afastamento evangélico).